# Module 7 – Discounting Cash Flows

## Objective

This notebook demonstrates how future cash flows are converted into present values using the bootstrapped zero-coupon yield curve.

Pipeline:

Market Quotes
→ Bootstrapper
→ Yield Curve
→ Schedule
→ Cash Flow Generation
→ Discounting
→ Present Value

Each cash flow is discounted independently using:

PV = CF × DF

where

- CF = Cash Flow
- DF = Discount Factor

In [1]:
from datetime import date

from src.cashflows.cashflow_generator import CashFlowGenerator
from src.common.enums import (
    DayCountConvention,
    FloatingIndex,
    PayReceive,
    PaymentFrequency,
)
from src.curves.bootstrapper import Bootstrapper
from src.market_data.market_instrument import MarketInstrument
from src.market_data.market_quote_collection import MarketQuoteCollection
from src.pricing.discounting import Discounting
from src.products.fixed_leg import FixedLeg
from src.products.floating_leg import FloatingLeg
from src.products.interest_rate_swap import InterestRateSwap
from src.products.trade import Trade
from src.schedule.schedule_generator import ScheduleGenerator

## Step 1

Construct the zero-coupon yield curve from market quotes.

In [2]:
quotes = MarketQuoteCollection()

quotes.add(MarketInstrument("Deposit", "1M", 0.048))
quotes.add(MarketInstrument("Deposit", "3M", 0.050))
quotes.add(MarketInstrument("Deposit", "6M", 0.052))
quotes.add(MarketInstrument("Deposit", "1Y", 0.055))
quotes.add(MarketInstrument("Deposit", "2Y", 0.057))
quotes.add(MarketInstrument("Deposit", "3Y", 0.059))

curve = Bootstrapper(
    valuation_date=date(2026, 1, 1),
    market_quotes=quotes,
).build()

curve.summary()

Yield Curve
Valuation Date : 2026-01-01

Tenor        Years       Zero Rate       Discount Factor
------------------------------------------------------------------------
1M          0.0833        4.8000%              0.996008
3M          0.2500        5.0000%              0.987578
6M          0.5000        5.2000%              0.974335
1Y          1.0000        5.5000%              0.946485
2Y          2.0000        5.7000%              0.892258
3Y          3.0000        5.9000%              0.837780
------------------------------------------------------------------------
Interpolation : Linear
Curve Points  : 6


## Step 2

Create a sample interest rate swap.

In [3]:
trade = Trade(
    trade_id="IRS001",
    counterparty="ABC Bank",
    currency="USD",
    effective_date=date(2026, 1, 1),
    maturity_date=date(2029, 1, 1),
)

fixed_leg = FixedLeg(
    notional=1_000_000,
    fixed_rate=0.05,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.RECEIVE,
)

floating_leg = FloatingLeg(
    notional=1_000_000,
    payment_frequency=PaymentFrequency.ANNUAL,
    day_count=DayCountConvention.ACT_365,
    pay_receive=PayReceive.PAY,
    floating_index=FloatingIndex.SOFR,
)

swap = InterestRateSwap(
    trade=trade,
    fixed_leg=fixed_leg,
    floating_leg=floating_leg,
)

## Step 3

Generate the fixed-leg cash flows.

In [4]:
schedule = ScheduleGenerator().generate(swap)

cashflows = CashFlowGenerator().generate(
    swap,
    schedule,
)

for cf in cashflows:
    print(cf)

CashFlow(payment_date=datetime.date(2027, 1, 1), accrual_start=datetime.date(2026, 1, 1), accrual_end=datetime.date(2027, 1, 1), year_fraction=1.0, notional=1000000, rate=0.05, amount=50000.0, currency='USD', cashflow_type=<CashFlowType.FIXED: 'Fixed'>)
CashFlow(payment_date=datetime.date(2028, 1, 1), accrual_start=datetime.date(2027, 1, 1), accrual_end=datetime.date(2028, 1, 1), year_fraction=1.0, notional=1000000, rate=0.05, amount=50000.0, currency='USD', cashflow_type=<CashFlowType.FIXED: 'Fixed'>)
CashFlow(payment_date=datetime.date(2029, 1, 1), accrual_start=datetime.date(2028, 1, 1), accrual_end=datetime.date(2029, 1, 1), year_fraction=1.0027397260273974, notional=1000000, rate=0.05, amount=50136.98630136987, currency='USD', cashflow_type=<CashFlowType.FIXED: 'Fixed'>)


## Step 4

Discount each cash flow using the yield curve.

In [5]:
discounted = Discounting().discount(
    cashflows,
    curve,
)

for dcf in discounted:
    print(dcf)

2027-01-01 | DF=0.946485 | CF=50,000.00 | PV=47,324.26
2028-01-01 | DF=0.892258 | CF=50,000.00 | PV=44,612.90
2029-01-01 | DF=0.837780 | CF=50,136.99 | PV=42,003.75


## Summary

This module introduced present value calculations.

Completed pipeline:

Market Quotes
→ Yield Curve
→ Schedule
→ Cash Flows
→ Discounted Cash Flows

The next module combines the discounted cash flows into a single Net Present Value (NPV) for the swap.